[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/091.%20The%20Contract%20Design%20to%20Mitigate%20the%20Bullwhip%20Effect/P91-Tier-6_executed.ipynb)

---

# 91. Contract Design to Mitigate the Bullwhip Effect
## Tier 6: Autonomous, Self-Optimizing Ecosystem

This notebook presents an illustrative multi-agent simulation of supply-chain participants negotiating contracts. Each agent senses a simplified operating environment, evaluates proposals, makes counter-proposals, and learns from negotiation outcomes. Agreements are recorded and executed by an in-memory smart-contract ledger.

### Objectives, inputs, and outputs

- **Objective:** demonstrate how autonomous negotiation over information sharing, capacity reservation, and penalty terms can be connected to a simplified bullwhip indicator.
- **Inputs:** agent capability ranges, the random seed, the number of agents and simulation periods, and the rules defined in the code.
- **Outputs:** negotiation and contract records, period-by-period system indicators, summary statistics, and visualizations.
- **Solution sequence:** define agents and their components; define the negotiation and contract-execution systems; simulate the supply chain; then analyze the recorded results.

### Model assumptions and scope

Agents are represented by simple rules and randomly initialized capabilities. Demand, inventory, utility, and contract effects are stylized quantities rather than calibrated estimates from a real supply chain. The bullwhip indicator in this notebook is a **proxy**, not the empirical bullwhip ratio defined as the variance of upstream orders divided by the variance of downstream demand. An accepted contract reduces the participating agents' modeled demand-variance values according to its terms; this assumption creates a direct, inspectable link between agreement and the proxy.

The blockchain component is an in-memory ledger that records simulated contract execution. It does not implement a distributed blockchain, cryptographic validation, legal enforcement, or real-time data integration. The simulation does not establish that the example performance figures sometimes associated with autonomous contracting will occur in practice; its outputs depend on the stated rules, parameters, and random seed.

### Negotiation model

A **proposal** is a candidate agreement from one agent to another. Each agent evaluates the other agent's proposal and returns a utility score between zero and one. In this model, an agreement is accepted only when both agents' evaluated utilities exceed the threshold \(\tau = 0.6\). If that condition is not met, agents adjust their proposals and continue for at most \(R = 10\) rounds.

For an agent evaluating a proposal, the utility calculation has the structure

$$
U = \operatorname{clip}_{[0,1]}\!\left[(U_0 + B - C) F_I F_D\right],
$$

where \(U_0 = 0.5\) is the base utility, \(B\) and \(C\) are contract-type-specific benefit and cost terms, and \(F_I\) and \(F_D\) adjust utility for inventory utilization and demand pressure. The function `EvaluationEngine.evaluate_contract` implements these calculations. The terms used for each contract type are specified in that function; not every term generated by a proposal is used in the utility calculation.

When both agents accept, the contract terms are the arithmetic means of the numeric terms in the two proposals. For an information-sharing agreement, the model updates each participant's demand-variance value as

$$
\sigma' = \max\!\left(0.01,\; \sigma\left(1 - 0.4s\right)\right),
$$

where \(\sigma\) is that agent's current demand-variance value and \(s\) is the agreed information-sharing level. Capacity-reservation and penalty-structure agreements use their corresponding terms and reduction factors in the negotiation code. These are illustrative behavioral rules, not empirically estimated causal effects.

The system-level proxy is

$$
B_t = 1 + 2\left(\frac{1}{N}\sum_{i=1}^{N}\sigma_{i,t}\right),
$$

where \(N\) is the number of agents and \(\sigma_{i,t}\) is agent \(i\)'s modeled demand variance at period \(t\). The implementation is `MultiAgentContractNegotiationSystem._calculate_system_metrics`. A lower proxy indicates lower average modeled variance under this particular rule; it should not be interpreted as a measured reduction in real order amplification.

### How to interpret the simulation

The simulation is a rule-based demonstration, not an optimization model: it has no mathematical-programming solver, global objective function, or guarantee of a globally optimal contract. Negotiation outcomes depend on proposal generation, evaluation, counter-proposal rules, and random initialization. The plots and summaries describe this run only. In particular, a successful simulated agreement demonstrates that the rules reached their acceptance condition; it does not by itself establish commercial feasibility or improved real-world performance.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

np.random.seed(91)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully for the multi-agent simulation.")

## 1. Agent architecture and contract proposals

An autonomous agent combines a state (such as inventory and demand), a sensing component, an evaluation engine, a proposal strategy, and a learning component. Separating these responsibilities makes the negotiation flow easier to inspect: sensing provides the state used by evaluation, strategy creates proposals, and learning can adjust the agent's negotiation style after repeated outcomes.

A contract proposal contains its proposer and target, contract type, numeric terms, priority, expiration period, and a short justification. In this simulation, priority, expiration, and justification are recorded proposal attributes; the agreement rule is based on evaluated utility.

In [ ]:
@dataclass
class ContractProposal:
    """A candidate set of contract terms sent from one agent to another."""

    proposer_id: str
    target_id: str
    contract_type: str
    terms: Dict[str, float]
    priority: float
    expiration_time: int
    justification: str


@dataclass
class SmartContract:
    """An agreed contract and its simulated execution records."""

    contract_id: str
    parties: List[str]
    terms: Dict[str, float]
    conditions: List[Dict]
    execution_history: List[Dict]
    performance_metrics: Dict[str, float]
    is_active: bool


class SensingModule:
    """Expose an agent's current state to its evaluation and strategy components."""

    def __init__(self, agent: "AutonomousAgent"):
        self.agent = agent

    def get_environment_state(self) -> Dict[str, float]:
        """Return the state variables used by proposal evaluation and generation."""
        return {
            "inventory_level": self.agent.current_inventory,
            "demand_level": self.agent.current_demand,
            "demand_variance": self.agent.demand_variance,
            "capacity_utilization": (
                self.agent.current_inventory / self.agent.capacity
                if self.agent.capacity > 0
                else 0.0
            ),
            "demand_pressure": (
                self.agent.current_demand / self.agent.capacity
                if self.agent.capacity > 0
                else 0.0
            ),
            "contract_load": float(len(self.agent.contracts)),
            "partner_satisfaction": self._estimate_partner_satisfaction(),
        }

    def _estimate_partner_satisfaction(self) -> float:
        """Use a simple estimate when partner-level observations are unavailable."""
        if not self.agent.partners:
            return 0.5
        return float(np.random.uniform(0.3, 0.8))


class EvaluationEngine:
    """Calculate an agent's utility for a candidate contract proposal."""

    def __init__(self, agent: "AutonomousAgent"):
        self.agent = agent

    def evaluate_contract(self, proposal: ContractProposal) -> Tuple[float, str]:
        """Return a clipped utility score and a concise account of its components."""
        environment = self.agent.sense_environment()
        utility = 0.5
        reasoning_parts = []

        if proposal.contract_type == "information_sharing":
            info_level = proposal.terms.get("info_sharing_level", 0.5)
            cost_share = proposal.terms.get("cost_allocation", 0.5)
            info_benefit = info_level * 0.3
            cost_penalty = cost_share * 0.2
            utility += info_benefit - cost_penalty
            reasoning_parts.append(
                f"Information benefit: {info_benefit:.3f}, "
                f"cost penalty: {cost_penalty:.3f}"
            )

        elif proposal.contract_type == "capacity_reservation":
            capacity_level = proposal.terms.get("capacity_commitment", 0.5)
            price_premium = proposal.terms.get("price_premium", 0.1)
            demand_pressure = environment["demand_pressure"]
            capacity_value = capacity_level * demand_pressure * 0.4
            premium_cost = price_premium * 0.3
            utility += capacity_value - premium_cost
            reasoning_parts.append(
                f"Capacity value: {capacity_value:.3f}, "
                f"premium cost: {premium_cost:.3f}"
            )

        elif proposal.contract_type == "penalty_structure":
            penalty_severity = proposal.terms.get("penalty_severity", 0.2)
            bonus_level = proposal.terms.get("performance_bonus", 0.1)
            reliability_bonus = self.agent.reliability * bonus_level * 0.2
            penalty_risk = (
                penalty_severity * (1 - self.agent.reliability) * 0.3
            )
            utility += reliability_bonus - penalty_risk
            reasoning_parts.append(
                f"Reliability bonus: {reliability_bonus:.3f}, "
                f"penalty risk: {penalty_risk:.3f}"
            )

        inventory_factor = (
            1 - abs(environment["capacity_utilization"] - 0.5) * 0.2
        )
        demand_factor = 1 + (environment["demand_pressure"] - 1) * 0.1
        utility *= inventory_factor * demand_factor
        utility = float(np.clip(utility, 0.0, 1.0))

        reasoning_parts.append(
            f"Inventory factor: {inventory_factor:.3f}, "
            f"demand factor: {demand_factor:.3f}"
        )
        return utility, " | ".join(reasoning_parts)


class StrategyManager:
    """Generate proposals using a cooperative, competitive, or adaptive style."""

    def __init__(self, agent: "AutonomousAgent"):
        self.agent = agent
        self.negotiation_style = str(
            np.random.choice(["cooperative", "competitive", "adaptive"])
        )
        self.risk_tolerance = float(np.random.uniform(0.3, 0.8))

    def generate_proposal(
        self, target_id: str, contract_type: str
    ) -> ContractProposal:
        """Construct a proposal from the agent's current state and negotiation style."""
        environment = self.agent.sense_environment()

        if contract_type == "information_sharing":
            terms = self._generate_info_sharing_terms(environment)
            justification = "Improve demand visibility and reduce variability."
        elif contract_type == "capacity_reservation":
            terms = self._generate_capacity_terms(environment)
            justification = "Secure capacity during periods of high demand."
        elif contract_type == "penalty_structure":
            terms = self._generate_penalty_terms(environment)
            justification = "Align incentives and improve service reliability."
        else:
            terms = {}
            justification = "General contract improvement."

        priority = self._calculate_proposal_priority(environment, contract_type)
        return ContractProposal(
            proposer_id=self.agent.agent_id,
            target_id=target_id,
            contract_type=contract_type,
            terms=terms,
            priority=priority,
            expiration_time=50,
            justification=justification,
        )

    def _generate_info_sharing_terms(self, environment: Dict) -> Dict[str, float]:
        """Set information-sharing terms in response to modeled demand variability."""
        base_sharing = 0.3 + environment["demand_variance"] * 0.4

        if self.negotiation_style == "cooperative":
            info_sharing_level = base_sharing + 0.2
            cost_allocation = 0.4
        elif self.negotiation_style == "competitive":
            info_sharing_level = base_sharing - 0.1
            cost_allocation = 0.7
        else:
            info_sharing_level = base_sharing + np.random.uniform(-0.1, 0.2)
            cost_allocation = 0.5 + np.random.uniform(-0.2, 0.2)

        return {
            "info_sharing_level": float(np.clip(info_sharing_level, 0.1, 1.0)),
            "cost_allocation": float(np.clip(cost_allocation, 0.0, 1.0)),
            "update_frequency": float(np.random.uniform(0.1, 1.0)),
            "data_quality_requirement": float(np.random.uniform(0.7, 1.0)),
        }

    def _generate_capacity_terms(self, environment: Dict) -> Dict[str, float]:
        """Set a capacity commitment and premium in response to demand pressure."""
        demand_pressure = environment["demand_pressure"]
        base_commitment = min(0.8, 0.3 + demand_pressure * 0.5)

        if self.negotiation_style == "cooperative":
            capacity_commitment = base_commitment + 0.1
            price_premium = 0.05
        elif self.negotiation_style == "competitive":
            capacity_commitment = base_commitment - 0.1
            price_premium = 0.15
        else:
            capacity_commitment = base_commitment + np.random.uniform(-0.1, 0.1)
            price_premium = 0.1 + np.random.uniform(-0.05, 0.05)

        return {
            "capacity_commitment": float(
                np.clip(capacity_commitment, 0.1, 1.0)
            ),
            "price_premium": float(np.clip(price_premium, 0.0, 0.3)),
            "flexibility_clause": float(np.random.uniform(0.2, 0.8)),
            "minimum_order_quantity": float(
                np.random.uniform(0.1, 0.5) * self.agent.capacity
            ),
        }

    def _generate_penalty_terms(self, environment: Dict) -> Dict[str, float]:
        """Set penalty and bonus terms in response to the agent's reliability."""
        base_severity = 0.2 * (1 - self.agent.reliability)

        if self.negotiation_style == "cooperative":
            penalty_severity = base_severity - 0.05
            performance_bonus = 0.15
        elif self.negotiation_style == "competitive":
            penalty_severity = base_severity + 0.05
            performance_bonus = 0.05
        else:
            penalty_severity = base_severity + np.random.uniform(-0.05, 0.05)
            performance_bonus = 0.1 + np.random.uniform(-0.05, 0.05)

        return {
            "penalty_severity": float(np.clip(penalty_severity, 0.05, 0.5)),
            "performance_bonus": float(np.clip(performance_bonus, 0.0, 0.3)),
            "response_time_requirement": float(np.random.uniform(0.1, 0.9)),
            "quality_threshold": float(np.random.uniform(0.8, 1.0)),
        }

    def _calculate_proposal_priority(
        self, environment: Dict, contract_type: str
    ) -> float:
        """Rank a proposal using a simple contract-specific urgency signal."""
        if contract_type == "information_sharing":
            urgency = environment["demand_variance"]
        elif contract_type == "capacity_reservation":
            urgency = max(0.0, environment["demand_pressure"] - 0.7)
        elif contract_type == "penalty_structure":
            urgency = 1 - environment["partner_satisfaction"]
        else:
            urgency = 0.3

        return float(np.clip(0.5 + urgency * 0.4, 0.1, 1.0))


class LearningComponent:
    """Track outcomes and adjust an agent's negotiation style when performance is low."""

    def __init__(self, agent: "AutonomousAgent"):
        self.agent = agent
        self.learning_rate = 0.1
        self.strategy_performance = defaultdict(list)

    def update_from_outcome(self, outcome: Dict) -> None:
        """Record an outcome and consider a style adjustment after repeated results."""
        utility = float(outcome.get("utility", 0.5))
        contract_type = outcome.get("contract_type", "unknown")
        self.strategy_performance[contract_type].append(utility)

        recent = self.strategy_performance[contract_type][-5:]
        if len(recent) == 5 and float(np.mean(recent)) < 0.4:
            self._adjust_strategy()

    def _adjust_strategy(self) -> None:
        """Move toward a more cooperative style after persistently low utility."""
        style = self.agent.strategy_manager.negotiation_style
        if style == "competitive":
            self.agent.strategy_manager.negotiation_style = "adaptive"
        elif style == "adaptive":
            self.agent.strategy_manager.negotiation_style = "cooperative"

        risk_tolerance = self.agent.strategy_manager.risk_tolerance
        if risk_tolerance > 0.7:
            self.agent.strategy_manager.risk_tolerance = max(
                0.3, risk_tolerance - self.learning_rate
            )
        elif risk_tolerance < 0.3:
            self.agent.strategy_manager.risk_tolerance = min(
                0.8, risk_tolerance + self.learning_rate
            )


class AutonomousAgent:
    """A supply-chain participant with state, sensing, evaluation, strategy, and learning."""

    def __init__(self, agent_id: str, agent_type: str, capabilities: Dict):
        self.agent_id = agent_id
        self.agent_type = agent_type

        self.capacity = float(capabilities.get("capacity", 1000))
        self.lead_time = float(capabilities.get("lead_time", 7))
        self.unit_cost = float(capabilities.get("unit_cost", 10))
        self.quality_score = float(capabilities.get("quality_score", 0.8))
        self.flexibility = float(capabilities.get("flexibility", 0.7))
        self.reliability = float(capabilities.get("reliability", 0.9))

        self.current_inventory = float(np.random.uniform(0.3, 0.8) * self.capacity)
        self.current_demand = float(np.random.uniform(0.5, 1.2) * self.capacity)
        self.demand_variance = float(np.random.uniform(0.1, 0.4))

        self.contracts = []
        self.partners = []
        self.performance_history = deque(maxlen=100)
        self.satisfaction_score = 0.5
        self.negotiation_history = []
        self.proposed_contracts = []
        self.received_proposals = []

        self.sensing_module = SensingModule(self)
        self.evaluation_engine = EvaluationEngine(self)
        self.strategy_manager = StrategyManager(self)
        self.learning_component = LearningComponent(self)

    def sense_environment(self) -> Dict[str, float]:
        return self.sensing_module.get_environment_state()

    def evaluate_proposal(self, proposal: ContractProposal) -> Tuple[float, str]:
        return self.evaluation_engine.evaluate_contract(proposal)

    def generate_proposal(
        self, target_id: str, contract_type: str
    ) -> ContractProposal:
        proposal = self.strategy_manager.generate_proposal(target_id, contract_type)
        self.proposed_contracts.append(proposal)
        return proposal

    def learn_from_outcome(self, outcome: Dict) -> None:
        self.learning_component.update_from_outcome(outcome)

    def update_performance(self, performance: Dict) -> None:
        self.performance_history.append(performance)
        recent = list(self.performance_history)[-10:]
        if recent:
            self.satisfaction_score = float(
                np.mean([item.get("utility", 0.5) for item in recent])
            )


print("Agent, proposal, and learning components are ready.")

## 2. Simulation, negotiation, and contract execution

The simulation creates agents with different capability ranges and runs negotiation rounds between randomly selected pairs. A negotiation begins with one proposal from each participant. In each round, both agents evaluate the other agent's proposal; if both scores exceed the acceptance threshold, the system averages the numeric terms, records a smart contract, and applies the model's contract effect. Otherwise, an agent with utility below the threshold adjusts its proposal toward the proposal it received.

The loop stops when an agreement is reached or the maximum number of rounds is exhausted. With at most \(R\) rounds and \(K\) numeric terms, proposal evaluation and adjustment take \(O(RK)\) work per pair, apart from fixed-size agent-state operations. The simulation repeats this process across selected pairs and periods; it does not search all possible contracts or guarantee convergence.

In [ ]:
class BlockchainSystem:
    """In-memory ledger for the simulation's smart-contract records and executions."""

    def __init__(self):
        self.smart_contracts = []
        self.execution_history = []
        self.contract_counter = 0

    def create_smart_contract(
        self, parties: List[str], terms: Dict, conditions: List[Dict]
    ) -> str:
        """Create an active contract record and return its unique identifier."""
        contract_id = f"SC_{self.contract_counter}"
        self.contract_counter += 1
        smart_contract = SmartContract(
            contract_id=contract_id,
            parties=list(parties),
            terms=dict(terms),
            conditions=list(conditions),
            execution_history=[],
            performance_metrics={
                "compliance_rate": 0.0,
                "execution_success": 0.0,
                "dispute_count": 0,
                "mean_utility": 0.0,
            },
            is_active=True,
        )
        self.smart_contracts.append(smart_contract)
        return contract_id

    def execute_contract(self, contract_id: str, execution_data: Dict) -> bool:
        """Record one successful simulated execution for an active contract."""
        contract = next(
            (
                item
                for item in self.smart_contracts
                if item.contract_id == contract_id
            ),
            None,
        )
        if contract is None or not contract.is_active:
            return False

        record = {
            "timestamp": execution_data.get("period", len(self.execution_history)),
            "execution_data": dict(execution_data),
            "success": True,
        }
        contract.execution_history.append(record)
        self.execution_history.append(record)
        contract.performance_metrics["execution_success"] = 1.0
        contract.performance_metrics["compliance_rate"] = 1.0
        contract.performance_metrics["mean_utility"] = float(
            np.mean(execution_data["utilities"])
        )
        return True


class NegotiationSystem:
    """Coordinate proposal evaluation, counter-proposals, and contract creation."""

    def __init__(
        self, agents: List[AutonomousAgent], blockchain: BlockchainSystem
    ):
        self.agents = agents
        self.blockchain = blockchain
        self.max_negotiation_rounds = 10
        self.acceptance_threshold = 0.6
        self.current_period = 0

    def run_negotiation_round(self) -> List[Dict]:
        """Run negotiations between randomly selected, distinct agent pairs."""
        results = []
        n_negotiations = min(5, len(self.agents) // 2)
        contract_types = [
            "information_sharing",
            "capacity_reservation",
            "penalty_structure",
        ]

        for _ in range(n_negotiations):
            indices = np.random.choice(len(self.agents), 2, replace=False)
            agent1 = self.agents[int(indices[0])]
            agent2 = self.agents[int(indices[1])]
            contract_type = str(np.random.choice(contract_types))
            results.append(
                self._negotiate_between_agents(agent1, agent2, contract_type)
            )
        return results

    def focused_negotiation(
        self,
        agent1: AutonomousAgent,
        agent2: AutonomousAgent,
        contract_type: str,
    ) -> Dict:
        """Run one measured negotiation between a specified pair of agents."""
        print(
            f"\nFocused negotiation: {agent1.agent_id} and {agent2.agent_id}"
        )
        print(f"Contract type: {contract_type}")
        start_time = time.perf_counter()
        result = self._negotiate_between_agents(agent1, agent2, contract_type)
        result["negotiation_time_ms"] = (
            time.perf_counter() - start_time
        ) * 1000
        print(f"Negotiation runtime: {result['negotiation_time_ms']:.1f} ms")
        return result

    def _negotiate_between_agents(
        self,
        agent1: AutonomousAgent,
        agent2: AutonomousAgent,
        contract_type: str,
    ) -> Dict:
        """Run the bounded proposal, evaluation, and counter-proposal process."""
        proposal1 = agent1.generate_proposal(agent2.agent_id, contract_type)
        proposal2 = agent2.generate_proposal(agent1.agent_id, contract_type)
        agreement_reached = False
        final_contract = None
        final_terms = {}
        utility1 = 0.0
        utility2 = 0.0
        negotiation_round = 0

        for round_num in range(self.max_negotiation_rounds):
            negotiation_round = round_num + 1
            utility1, _ = agent1.evaluate_proposal(proposal2)
            utility2, _ = agent2.evaluate_proposal(proposal1)

            if (
                utility1 > self.acceptance_threshold
                and utility2 > self.acceptance_threshold
            ):
                agreement_reached = True
                final_terms = self._create_hybrid_terms(
                    proposal1.terms, proposal2.terms
                )
                parties = [agent1.agent_id, agent2.agent_id]
                conditions = [
                    {"type": "performance_monitoring", "threshold": 0.8},
                    {"type": "renegotiation_trigger", "variance_threshold": 0.3},
                ]
                contract_id = self.blockchain.create_smart_contract(
                    parties, final_terms, conditions
                )
                final_contract = {
                    "contract_id": contract_id,
                    "parties": parties,
                    "contract_type": contract_type,
                    "terms": final_terms,
                    "utilities": (utility1, utility2),
                }

                agent1.contracts.append(final_contract)
                agent2.contracts.append(final_contract)
                if agent2.agent_id not in agent1.partners:
                    agent1.partners.append(agent2.agent_id)
                if agent1.agent_id not in agent2.partners:
                    agent2.partners.append(agent1.agent_id)

                self.blockchain.execute_contract(
                    contract_id,
                    {
                        "period": self.current_period,
                        "contract_type": contract_type,
                        "utilities": (utility1, utility2),
                    },
                )
                self._apply_contract_effect(
                    agent1, agent2, contract_type, final_terms
                )
                break

            if round_num < self.max_negotiation_rounds - 1:
                if utility1 < self.acceptance_threshold:
                    proposal1 = self._adjust_proposal(
                        agent1, proposal2, utility1
                    )
                if utility2 < self.acceptance_threshold:
                    proposal2 = self._adjust_proposal(
                        agent2, proposal1, utility2
                    )

        bullwhip_reduction = 0.0
        if agreement_reached:
            if contract_type == "information_sharing":
                bullwhip_reduction = (
                    final_terms.get("info_sharing_level", 0.5) * 0.4
                )
            elif contract_type == "capacity_reservation":
                bullwhip_reduction = (
                    final_terms.get("capacity_commitment", 0.5) * 0.3
                )
            elif contract_type == "penalty_structure":
                bullwhip_reduction = (
                    final_terms.get("performance_bonus", 0.1) * 0.2
                )

        outcome = {
            "agent1_id": agent1.agent_id,
            "agent2_id": agent2.agent_id,
            "contract_type": contract_type,
            "success": agreement_reached,
            "rounds": negotiation_round,
            "final_contract": final_contract,
            "bullwhip_reduction": bullwhip_reduction,
            "agent1_utility": utility1 if agreement_reached else 0.0,
            "agent2_utility": utility2 if agreement_reached else 0.0,
        }

        agent1.update_performance({"utility": outcome["agent1_utility"]})
        agent2.update_performance({"utility": outcome["agent2_utility"]})
        agent1.learn_from_outcome(
            {
                "utility": outcome["agent1_utility"],
                "contract_type": contract_type,
                "success": agreement_reached,
            }
        )
        agent2.learn_from_outcome(
            {
                "utility": outcome["agent2_utility"],
                "contract_type": contract_type,
                "success": agreement_reached,
            }
        )
        agent1.negotiation_history.append(outcome)
        agent2.negotiation_history.append(outcome)
        return outcome

    def _apply_contract_effect(
        self,
        agent1: AutonomousAgent,
        agent2: AutonomousAgent,
        contract_type: str,
        terms: Dict[str, float],
    ) -> None:
        """Apply the simulation's stated contract-to-variance rule to both parties."""
        if contract_type == "information_sharing":
            reduction_factor = 0.4 * terms.get("info_sharing_level", 0.5)
        elif contract_type == "capacity_reservation":
            reduction_factor = 0.3 * terms.get("capacity_commitment", 0.5)
        elif contract_type == "penalty_structure":
            reduction_factor = 0.2 * terms.get("performance_bonus", 0.1)
        else:
            reduction_factor = 0.0

        reduction_factor = float(np.clip(reduction_factor, 0.0, 1.0))
        for agent in (agent1, agent2):
            agent.demand_variance = max(
                0.01, agent.demand_variance * (1 - reduction_factor)
            )

    def _create_hybrid_terms(self, terms1: Dict, terms2: Dict) -> Dict[str, float]:
        """Average matching numeric proposal terms to form the agreed terms."""
        hybrid_terms = {}
        for key in set(terms1) | set(terms2):
            value1 = terms1.get(key, 0.0)
            value2 = terms2.get(key, 0.0)
            hybrid_terms[key] = (float(value1) + float(value2)) / 2
        return hybrid_terms

    def _adjust_proposal(
        self,
        agent: AutonomousAgent,
        received_proposal: ContractProposal,
        utility: float,
    ) -> ContractProposal:
        """Move the agent's newly generated terms toward the proposal it received."""
        current_proposal = agent.generate_proposal(
            received_proposal.target_id, received_proposal.contract_type
        )
        compromise_factor = float(np.clip(1 - utility, 0.0, 1.0))
        new_terms = {}

        for key, received_value in received_proposal.terms.items():
            current_value = current_proposal.terms.get(key, 0.5)
            new_terms[key] = current_value + (
                received_value - current_value
            ) * compromise_factor

        return ContractProposal(
            proposer_id=agent.agent_id,
            target_id=received_proposal.proposer_id,
            contract_type=received_proposal.contract_type,
            terms=new_terms,
            priority=received_proposal.priority * 0.9,
            expiration_time=max(0, received_proposal.expiration_time - 5),
            justification="Counter-proposal based on negotiation feedback.",
        )


class MultiAgentContractNegotiationSystem:
    """Create agents, simulate negotiations, and collect system-level indicators."""

    def __init__(self, n_agents: int = 8, n_simulation_periods: int = 100):
        if n_agents < 2:
            raise ValueError("At least two agents are required for negotiation.")
        if n_simulation_periods < 1:
            raise ValueError("At least one simulation period is required.")

        self.n_agents = n_agents
        self.n_simulation_periods = n_simulation_periods
        self.agents = self._create_agents()
        self.blockchain = BlockchainSystem()
        self.negotiation_system = NegotiationSystem(self.agents, self.blockchain)
        self.current_period = 0
        self.simulation_history = []
        self.contract_modification_history = []
        self.negotiation_outcomes = []
        self.bullwhip_history = []

    def _create_agents(self) -> List[AutonomousAgent]:
        """Create agents with the model's fixed role distribution and capability ranges."""
        agent_types = ["supplier", "manufacturer", "retailer", "logistics"]
        type_distribution = [3, 2, 2, 1]
        agents = []
        agent_id = 0

        for agent_type, count in zip(agent_types, type_distribution):
            for _ in range(count):
                capabilities = self._generate_agent_capabilities(agent_type)
                agents.append(
                    AutonomousAgent(
                        f"Agent_{agent_id}", agent_type, capabilities
                    )
                )
                agent_id += 1

        return agents[: self.n_agents]

    def _generate_agent_capabilities(self, agent_type: str) -> Dict[str, float]:
        """Draw capabilities from role-specific ranges used in this example."""
        ranges = {
            "supplier": {
                "capacity": (500, 2000),
                "lead_time": (7, 21),
                "unit_cost": (5, 25),
                "quality_score": (0.7, 0.95),
                "flexibility": (0.3, 0.8),
                "reliability": (0.8, 0.98),
            },
            "manufacturer": {
                "capacity": (800, 3000),
                "lead_time": (3, 14),
                "unit_cost": (15, 50),
                "quality_score": (0.8, 0.98),
                "flexibility": (0.4, 0.9),
                "reliability": (0.85, 0.99),
            },
            "retailer": {
                "capacity": (1000, 5000),
                "lead_time": (1, 7),
                "unit_cost": (2, 15),
                "quality_score": (0.75, 0.95),
                "flexibility": (0.5, 0.9),
                "reliability": (0.9, 0.99),
            },
            "logistics": {
                "capacity": (2000, 10000),
                "lead_time": (1, 5),
                "unit_cost": (1, 10),
                "quality_score": (0.85, 0.99),
                "flexibility": (0.6, 0.95),
                "reliability": (0.9, 0.99),
            },
        }
        if agent_type not in ranges:
            raise ValueError(f"Unsupported agent type: {agent_type}")
        return {
            name: float(np.random.uniform(low, high))
            for name, (low, high) in ranges[agent_type].items()
        }

    def simulate_critical_incident(self, period: int):
        """Apply the specified demand-variability shock at simulation period 30."""
        if period != 30:
            return False, None, None

        manufacturer = next(
            (agent for agent in self.agents if agent.agent_type == "manufacturer"),
            None,
        )
        supplier = next(
            (agent for agent in self.agents if agent.agent_type == "supplier"),
            None,
        )
        if manufacturer is None or supplier is None:
            return False, None, None

        print("\nCritical incident: modeled demand variability rises by 23%.")
        manufacturer.demand_variance *= 1.23
        manufacturer.current_demand *= 1.15
        print(
            f"{manufacturer.agent_id} demand variance: "
            f"{manufacturer.demand_variance:.3f}"
        )
        print(
            f"{manufacturer.agent_id} begins negotiation with "
            f"{supplier.agent_id}."
        )
        return True, manufacturer, supplier

    def run_simulation(self) -> List[Dict]:
        """Run the configured periods, negotiating and recording metrics each period."""
        print("=== MULTI-AGENT CONTRACT NEGOTIATION SIMULATION ===")
        print(
            f"Agents: {self.n_agents}; "
            f"simulation periods: {self.n_simulation_periods}"
        )

        for period in range(self.n_simulation_periods):
            self.current_period = period
            self.negotiation_system.current_period = period
            incident_detected, manufacturer, supplier = (
                self.simulate_critical_incident(period)
            )

            if incident_detected:
                result = self.negotiation_system.focused_negotiation(
                    manufacturer, supplier, "information_sharing"
                )
                self.negotiation_outcomes.append(result)
                if result["success"]:
                    print(
                        "Agreement reached in "
                        f"{result['rounds']} rounds; modeled proxy change "
                        f"factor: {result['bullwhip_reduction']:.3f}."
                    )
                else:
                    print("The focused negotiation did not reach agreement.")
            else:
                results = self.negotiation_system.run_negotiation_round()
                self.negotiation_outcomes.extend(results)

            self._update_agent_states()
            metrics = self._calculate_system_metrics()
            self.simulation_history.append(metrics)
            self.bullwhip_history.append(metrics["bullwhip_amplification"])
            self.contract_modification_history.append(
                metrics["contract_execution_rate"]
            )

            if (period + 1) % 20 == 0:
                print(
                    f"Period {period + 1}: "
                    f"{metrics['active_contracts']} active contracts; "
                    f"bullwhip proxy {metrics['bullwhip_amplification']:.3f}."
                )

        print("\nSimulation completed.")
        return self.simulation_history

    def _update_agent_states(self) -> None:
        """Advance demand and inventory using the simulation's period rules."""
        for agent in self.agents:
            demand_change = np.random.normal(0, 0.1)
            agent.current_demand *= 1 + demand_change
            agent.current_demand = max(
                0.1 * agent.capacity, agent.current_demand
            )

            agent.current_inventory -= agent.current_demand * 0.1
            agent.current_inventory = max(0.0, agent.current_inventory)
            if self.current_period % 10 == 0:
                agent.current_inventory = min(
                    agent.capacity,
                    agent.current_inventory + agent.capacity * 0.3,
                )

    def _calculate_system_metrics(self) -> Dict:
        """Calculate contract counts, satisfaction, and the stated variance proxy."""
        unique_contracts = {
            contract["contract_id"]: contract
            for agent in self.agents
            for contract in agent.contracts
        }
        active_contracts = len(unique_contracts)
        avg_satisfaction = float(
            np.mean([agent.satisfaction_score for agent in self.agents])
        )
        mean_demand_variance = float(
            np.mean([agent.demand_variance for agent in self.agents])
        )
        bullwhip_amplification = 1 + 2 * mean_demand_variance
        if self.current_period > 0:
            execution_rate = (
                len(self.blockchain.execution_history) / self.current_period
            )
        else:
            execution_rate = 0.0

        return {
            "period": self.current_period,
            "active_contracts": active_contracts,
            "avg_satisfaction": avg_satisfaction,
            "bullwhip_amplification": bullwhip_amplification,
            "contract_execution_rate": execution_rate,
            "smart_contracts_created": len(self.blockchain.smart_contracts),
        }

    def analyze_results(self) -> Optional[Dict]:
        """Summarize the recorded indicators without treating the proxy as empirical data."""
        print("\n=== SIMULATION RESULTS ANALYSIS ===")
        if not self.simulation_history:
            print("No simulation data are available.")
            return None

        periods = [item["period"] for item in self.simulation_history]
        contracts = [item["active_contracts"] for item in self.simulation_history]
        satisfaction = [item["avg_satisfaction"] for item in self.simulation_history]
        bullwhip = [
            item["bullwhip_amplification"] for item in self.simulation_history
        ]
        execution_rates = [
            item["contract_execution_rate"] for item in self.simulation_history
        ]

        print(f"Recorded simulation periods: {len(periods)}")
        print(f"Final active contracts: {contracts[-1]}")
        print(f"Final mean agent satisfaction: {satisfaction[-1]:.3f}")
        print(f"Final bullwhip proxy: {bullwhip[-1]:.3f}")
        print(
            "Mean contract executions per period: "
            f"{np.mean(execution_rates):.2f}"
        )
        print(
            "Smart contracts created: "
            f"{len(self.blockchain.smart_contracts)}"
        )

        initial_proxy = bullwhip[0]
        final_proxy = bullwhip[-1]
        reduction_percentage = (
            (initial_proxy - final_proxy) / initial_proxy * 100
            if initial_proxy != 0
            else 0.0
        )
        print(
            "Change in bullwhip proxy from first to final recorded period: "
            f"{reduction_percentage:.1f}%"
        )

        if len(bullwhip) > 35:
            pre_incident = float(np.mean(bullwhip[25:30]))
            post_incident = float(np.mean(bullwhip[30:35]))
            incident_change = (
                (pre_incident - post_incident) / pre_incident * 100
                if pre_incident != 0
                else 0.0
            )
            print(
                "Change in proxy across the selected incident window: "
                f"{incident_change:.1f}%"
            )

        return {
            "periods": periods,
            "contracts": contracts,
            "satisfaction": satisfaction,
            "bullwhip": bullwhip,
            "execution_rates": execution_rates,
            "reduction_percentage": reduction_percentage,
        }


print("Simulation, negotiation, and contract-ledger classes are ready.")

## 3. Configure and run the simulation

The parameters below retain the example's eight agents and one hundred periods. The fixed NumPy seed makes random draws reproducible when the notebook is run from the beginning in a compatible environment. A seed does not make wall-clock runtime measurements reproducible, and results can vary across library versions.

In [ ]:
mas_system = MultiAgentContractNegotiationSystem(
    n_agents=8,
    n_simulation_periods=100,
)

print("=== MULTI-AGENT CONTRACT NEGOTIATION SYSTEM ===")
print(f"Number of agents: {mas_system.n_agents}")
print(f"Simulation periods: {mas_system.n_simulation_periods}")
print(
    "Maximum negotiation rounds: "
    f"{mas_system.negotiation_system.max_negotiation_rounds}"
)

agent_types = defaultdict(list)
for agent in mas_system.agents:
    agent_types[agent.agent_type].append(agent.agent_id)

print("\nAGENT CONFIGURATION")
for agent_type, agent_ids in agent_types.items():
    print(f"{agent_type.capitalize()}: {', '.join(agent_ids)}")

print("\nSAMPLE AGENT CAPABILITIES")
for agent in mas_system.agents[:3]:
    print(f"\n{agent.agent_id} ({agent.agent_type}):")
    print(f"  Capacity: {agent.capacity:.0f}")
    print(f"  Lead time: {agent.lead_time:.1f} days")
    print(f"  Unit cost: ${agent.unit_cost:.2f}")
    print(f"  Quality score: {agent.quality_score:.3f}")
    print(f"  Negotiation style: {agent.strategy_manager.negotiation_style}")

simulation_results = mas_system.run_simulation()

## 4. Analyze and visualize system indicators

The plots below show simulated contract counts, the modeled bullwhip proxy, satisfaction, contract executions, and agent composition. The incident marker identifies the period at which the specified demand-variability shock is applied. It marks timing only: it does not imply that any observed change is statistically significant or caused solely by the incident response.

In [ ]:
analysis_results = mas_system.analyze_results()
if analysis_results is None:
    raise RuntimeError("Run the simulation before analyzing its results.")

periods = analysis_results["periods"]
contracts = analysis_results["contracts"]
bullwhip = analysis_results["bullwhip"]
satisfaction = analysis_results["satisfaction"]
execution_rates = analysis_results["execution_rates"]

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle(
    "Multi-Agent Contract Negotiation Simulation",
    fontsize=16,
    fontweight="bold",
)

ax1 = axes[0, 0]
ax1.plot(periods, contracts, color="tab:blue", marker="o", markersize=3)
ax1.set_xlabel("Simulation period")
ax1.set_ylabel("Unique active contracts")
ax1.set_title("Contract count")
ax1.grid(True, alpha=0.3)
ax1.axvline(30, color="tab:red", linestyle="--", label="Incident period")
ax1.legend()

ax2 = axes[0, 1]
ax2.plot(periods, bullwhip, color="tab:red", marker="s", markersize=3)
ax2.set_xlabel("Simulation period")
ax2.set_ylabel("Bullwhip proxy (index)")
ax2.set_title("Modeled demand-variance proxy")
ax2.grid(True, alpha=0.3)
ax2.axvline(30, color="tab:red", linestyle="--", label="Incident period")
ax2.legend()

ax3 = axes[0, 2]
ax3.plot(periods, satisfaction, color="tab:green", marker="^", markersize=3)
ax3.set_xlabel("Simulation period")
ax3.set_ylabel("Mean satisfaction (0–1)")
ax3.set_title("Mean agent satisfaction")
ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3)
ax3.axvline(30, color="tab:red", linestyle="--", label="Incident period")
ax3.legend()

ax4 = axes[1, 0]
ax4.plot(periods, execution_rates, color="tab:purple", marker="d", markersize=3)
ax4.set_xlabel("Simulation period")
ax4.set_ylabel("Executions per period")
ax4.set_title("Cumulative contract execution rate")
ax4.grid(True, alpha=0.3)
ax4.axvline(30, color="tab:red", linestyle="--", label="Incident period")
ax4.legend()

ax5 = axes[1, 1]
type_counts = {name: len(ids) for name, ids in agent_types.items()}
bars = ax5.bar(type_counts.keys(), type_counts.values(), color=sns.color_palette("Set2", len(type_counts)))
ax5.set_xlabel("Agent type")
ax5.set_ylabel("Number of agents")
ax5.set_title("Agent type distribution")
ax5.grid(True, axis="y", alpha=0.3)
for bar, count in zip(bars, type_counts.values()):
    ax5.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(count),
        ha="center",
        va="bottom",
    )

ax6 = axes[1, 2]
style_counts = defaultdict(int)
for agent in mas_system.agents:
    style_counts[agent.strategy_manager.negotiation_style] += 1
ax6.pie(
    style_counts.values(),
    labels=style_counts.keys(),
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("Set2", len(style_counts)),
)
ax6.set_title("Final negotiation-style distribution")

ax7 = axes[2, 0]
execution_periods = [
    record["execution_data"].get("period", record["timestamp"])
    for record in mas_system.blockchain.execution_history
]
if execution_periods:
    ax7.hist(
        execution_periods,
        bins=min(20, max(1, len(set(execution_periods)))),
        color="tab:orange",
        alpha=0.75,
        edgecolor="black",
    )
    ax7.axvline(30, color="tab:red", linestyle="--", label="Incident period")
    ax7.legend()
else:
    ax7.text(0.5, 0.5, "No contracts executed", ha="center", va="center")
ax7.set_xlabel("Simulation period")
ax7.set_ylabel("Number of executions")
ax7.set_title("Contract execution periods")
ax7.grid(True, alpha=0.3)

ax8 = axes[2, 1]
summary_labels = [
    "Contracts",
    "Bullwhip proxy",
    "Satisfaction",
    "Executions / period",
]
summary_values = [
    contracts[-1],
    bullwhip[-1],
    satisfaction[-1],
    float(np.mean(execution_rates)),
]
summary_scales = [
    max(contracts) if max(contracts) > 0 else 1,
    max(bullwhip) if max(bullwhip) > 0 else 1,
    1.0,
    max(execution_rates) if max(execution_rates) > 0 else 1,
]
normalized_values = [
    100 * value / scale
    for value, scale in zip(summary_values, summary_scales)
]
bars = ax8.bar(
    summary_labels,
    normalized_values,
    color=sns.color_palette("Set2", len(summary_labels)),
)
ax8.set_ylabel("Within-metric reference score (%)")
ax8.set_title("Final indicators (separately scaled)")
ax8.set_ylim(0, 115)
ax8.tick_params(axis="x", rotation=25)
ax8.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, summary_values):
    ax8.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{value:.2f}",
        ha="center",
        va="bottom",
    )

ax9 = axes[2, 2]
incident_indices = [index for index, period in enumerate(periods) if 25 <= period <= 35]
if incident_indices:
    incident_periods = [periods[index] for index in incident_indices]
    incident_values = [bullwhip[index] for index in incident_indices]
    ax9.plot(
        incident_periods,
        incident_values,
        color="tab:red",
        marker="o",
        linewidth=2,
        label="Bullwhip proxy",
    )
    ax9.axvline(30, color="black", linestyle="--", label="Incident period")
    ax9.legend()
else:
    ax9.text(0.5, 0.5, "Incident window is outside this run", ha="center", va="center")
ax9.set_xlabel("Simulation period")
ax9.set_ylabel("Bullwhip proxy (index)")
ax9.set_title("Incident-window view")
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Visualization complete.")

## 5. Inspect contracts, outcomes, and agent states

The following summaries distinguish contracts created from negotiation attempts: only accepted agreements appear in the contract ledger. The ledger records one simulated execution when an agreement is created. Consequently, its execution rate is a count-based simulation metric, not a measured rate of operational compliance over time.

In [ ]:
print("=== DETAILED NEGOTIATION ANALYSIS ===\n")
contracts = mas_system.blockchain.smart_contracts
executions = mas_system.blockchain.execution_history
print(f"Smart contracts created: {len(contracts)}")
print(f"Successful simulated executions: {len(executions)}")

if contracts:
    contract_types = defaultdict(int)
    contract_utilities = []
    for contract in contracts:
        contract_type = next(
            (
                key
                for key in (
                    "info_sharing_level",
                    "capacity_commitment",
                    "penalty_severity",
                )
                if key in contract.terms
            ),
            "other",
        )
        type_names = {
            "info_sharing_level": "information_sharing",
            "capacity_commitment": "capacity_reservation",
            "penalty_severity": "penalty_structure",
        }
        contract_types[type_names.get(contract_type, "other")] += 1
        contract_utilities.append(
            contract.performance_metrics["mean_utility"]
        )

    print("\nContract type distribution:")
    for contract_type, count in sorted(contract_types.items()):
        percentage = count / len(contracts) * 100
        print(f"  {contract_type}: {count} ({percentage:.1f}%)")
    print(f"Mean accepted utility across contracts: {np.mean(contract_utilities):.3f}")
    execution_percentage = len(executions) / len(contracts) * 100
    print(f"Contracts with a recorded execution: {execution_percentage:.1f}%")
else:
    print("No agreements were reached in this run.")

print("\nAGENT PERFORMANCE")
for agent in mas_system.agents:
    print(f"\n{agent.agent_id} ({agent.agent_type}):")
    print(f"  Satisfaction score: {agent.satisfaction_score:.3f}")
    print(f"  Contract records: {len(agent.contracts)}")
    print(f"  Trading partners: {len(agent.partners)}")
    print(f"  Negotiation style: {agent.strategy_manager.negotiation_style}")
    print(f"  Risk tolerance: {agent.strategy_manager.risk_tolerance:.3f}")
    if agent.performance_history:
        recent_performance = list(agent.performance_history)[-5:]
        recent_utility = np.mean(
            [item.get("utility", 0.5) for item in recent_performance]
        )
        print(f"  Recent mean utility: {recent_utility:.3f}")

attempted_negotiations = len(mas_system.negotiation_outcomes)
successful_negotiations = sum(
    outcome["success"] for outcome in mas_system.negotiation_outcomes
)
success_rate = (
    successful_negotiations / attempted_negotiations * 100
    if attempted_negotiations
    else 0.0
)
print("\nSYSTEM COUNTS")
print(f"Negotiations attempted: {attempted_negotiations}")
print(f"Negotiations reaching agreement: {successful_negotiations}")
print(f"Agreement rate: {success_rate:.1f}%")
print(
    "Mean unique contracts per agent: "
    f"{len(contracts) / len(mas_system.agents):.2f}"
)
print(
    "Mean recorded executions per period: "
    f"{np.mean(execution_rates):.2f}"
)
print(
    "Final bullwhip proxy: "
    f"{analysis_results['bullwhip'][-1]:.3f}"
)

## 6. Interpreting the tier and its limitations

### What this tier represents

This tier represents a rule-based form of autonomous coordination: agents generate proposals, evaluate incoming terms, counter-propose, and record accepted agreements. Its distinguishing feature in this example is that these steps are modeled as agent actions rather than as decisions made by a human during each simulated negotiation.

### Potential advantages and trade-offs

- **Potential advantage:** repeated proposal evaluation can represent rapid coordination when operating conditions change.
- **Potential advantage:** an explicit ledger provides a structured record of simulated agreements and executions.
- **Potential advantage:** agent-specific state and rules make it possible to study how local decisions interact at system level.
- **Trade-off:** agent behavior is only as credible as the rules and data used to specify it.
- **Trade-off:** pairwise negotiations and repeated evaluations create computational work as the number of agents, pairs, and rounds increases.
- **Trade-off:** local utility improvements do not guarantee system-wide performance improvements, fairness, or feasible contracts.
- **Trade-off:** an in-memory ledger does not provide the security, decentralization, or legal enforceability of production blockchain infrastructure.

### When this modeling approach may be useful

A multi-agent simulation can be useful for exploring how decentralized decision rules might interact, identifying assumptions that need testing, and generating questions for data collection or pilot studies. It is not a substitute for empirical validation, governance design, legal review, or operational testing. The simple satisfaction score and variance proxy in this notebook should be interpreted only according to their code definitions.

### Further investigation

Natural extensions include testing additional agent roles, adding constraints that reflect operational feasibility, calibrating utility and variance rules against observed data, and examining sensitivity to model assumptions. Such work should report its data, assumptions, and uncertainty; none of those extensions is implemented here.